# Phần B, bước 1: đầu ra UniAV từ Athena sạch (`961f1a89`)

Theo `docs/UEMR_B_CHAY_LAI_AZ.md`. Notebook này **chỉ chạy UniAV** (runtime riêng: `requirements_uniav.txt` không được cài chung
phiên với Omni). Bước 2 là `uemr_final_AZ_clean.ipynb` (runtime A100 khác), đọc đầu ra của notebook này.

| Ô | Việc |
|---|---|
| 1 | đường dẫn, kiểm file trên Drive; đầu ra đã có thì các ô sau tự bỏ qua |
| 2 | code Fushion, cài `requirements_uniav.txt`, `pytest uniav_app/tests` |
| 3 | checkpoint mới: sha256, nạp **strict** bằng `uniav_app` |
| 4 | 498 video gallery (394 val + 104 dev) + 1.002 video trainsplit, GT, feature HF |
| 5 | tái lập `final_eval` của checkpoint **trên dev** (trọng số chính, như `train.py`; lệch ≤ 1 điểm) |
| 6 | detector (`state_dict_seg`) cho 498 video: proposal thô + Z |
| 7 | θ chọn theo quy tắc protocol **chỉ trên 104 video dev** |
| 8 | hậu xử lý → `segments_pred.jsonl` gallery |
| 9 | như 6 + 8 cho 1.002 video trainsplit (cùng θ) |
| 10 | kiểm, `SOURCE.json`, chép lên Drive |

**Runtime:** T4 là đủ (khoảng 30–45 phút). **Run all.** Bị ngắt thì Run all lại (đầu ra chỉ chép lên Drive ở ô 10, khi đã đủ).

**Đầu ra** (không đụng `uniav_cache/ft`, `ft_train` của Athena cũ): `MyDrive/Colab Notebooks/code KL/Fushion/uemr/uniav_cache/athena_961f1a89/`
(gallery) và `athena_961f1a89_train/` (trainsplit): `segments_pred.jsonl`, `Z/`, `theta.json`, `raw_proposals.jsonl`, `predictions*.jsonl`, `SOURCE.json`.

In [1]:
# ===== 1. Drive, đường dẫn =====
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, glob, hashlib, shutil, subprocess, time, platform
from collections import Counter

FUSHION_GIT = 'https://github.com/chien24/Fushion_UEMR.git'
FUSHION     = '/content/code/Fushion_UEMR'
BASE        = '/content/drive/MyDrive/Colab Notebooks/code KL/Fushion/uemr'
U           = f'{BASE}/uniav_cache'
SUBSET      = f'{BASE}/omni_cache/common/subset_videos.json'
DRIVE_DATA  = '/content/drive/MyDrive/Colab Notebooks/code KL/Omni/data/YouCookII'
META        = f'{DRIVE_DATA}/metadata'
ATHENA_CKPT = f'{BASE}/athena_sach/961f1a89/athena_yc2clean.pth'
ATHENA_SHA  = '961f1a895f4b43c811d3e0cfa31448641989a0a0e39e421ed397682407643d58'
ATHENA_H8   = ATHENA_SHA[:8]
ATHENA_RUN  = 'yc2clean_E1'
LOCAL       = '/content/uniav_local'                       # chạy ở đĩa local (nhanh), ô 10 chép lên Drive
OUT_G, OUT_T = f'{U}/athena_{ATHENA_H8}', f'{U}/athena_{ATHENA_H8}_train'

def sha256(p, n=1 << 24):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(n), b''):
            h.update(b)
    return h.hexdigest()

for p in (ATHENA_CKPT, SUBSET, f'{META}/train_omni_video.trainsplit.jsonl', f'{U}/ft_train/durations.json'):
    print('%-6s %s' % ('OK' if os.path.exists(p) else 'THIẾU', p))
    assert os.path.exists(p), p
def delivered(d):
    p = f'{d}/SOURCE.json'
    return os.path.exists(p) and json.load(open(p)).get('sha256') == ATHENA_SHA
DONE = delivered(OUT_G) and delivered(OUT_T)
print('đầu ra đã có trên Drive -> các ô 4-10 bỏ qua' if DONE else 'chưa có đầu ra: chạy hết')

Mounted at /content/drive
OK     /content/drive/MyDrive/Colab Notebooks/code KL/Fushion/uemr/athena_sach/961f1a89/athena_yc2clean.pth
OK     /content/drive/MyDrive/Colab Notebooks/code KL/Fushion/uemr/omni_cache/common/subset_videos.json
OK     /content/drive/MyDrive/Colab Notebooks/code KL/Omni/data/YouCookII/metadata/train_omni_video.trainsplit.jsonl
OK     /content/drive/MyDrive/Colab Notebooks/code KL/Fushion/uemr/uniav_cache/ft_train/durations.json
chưa có đầu ra: chạy hết


In [2]:
# ===== 2. Code Fushion, cài đặt, unit test =====
%cd /content
os.makedirs('/content/code', exist_ok=True)
shutil.rmtree(FUSHION, ignore_errors=True)
!git clone -q {FUSHION_GIT} {FUSHION}
COMMIT = subprocess.run(['git', '-C', FUSHION, 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip()
if FUSHION not in sys.path:
    sys.path.insert(0, FUSHION)
os.environ['PYTHONPATH'] = FUSHION + ':' + os.environ.get('PYTHONPATH', '')
!pip install -q --only-binary numpy -r {FUSHION}/requirements_uniav.txt
%cd {FUSHION}
import numpy as np, pandas as pd, torch
from IPython.display import display
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'không có'
ENV = {'python': platform.python_version(), 'torch': torch.__version__, 'numpy': np.__version__, 'gpu': gpu, 'code': COMMIT}
print(json.dumps(ENV, indent=1, ensure_ascii=False))
r = subprocess.run([sys.executable, '-m', 'pytest', 'uniav_app/tests', '-q', '-p', 'no:cacheprovider'], cwd=FUSHION,
                   capture_output=True, text=True)
print(r.stdout[-1200:])
assert r.returncode == 0, 'unit test uniav_app chưa qua: dừng'
assert int(np.__version__.split('.')[0]) >= 2, 'checkpoint Athena cần numpy 2.x: Runtime > Restart session rồi chạy lại ô 1-2'

/content
/content/code/Fushion_UEMR
{
 "python": "3.13.15",
 "torch": "2.11.0+cu130",
 "numpy": "2.1.3",
 "gpu": "Tesla T4",
 "code": "545b861 config"
}
......................................                                   [100%]
38 passed in 2.27s



In [3]:
# ===== 3. Checkpoint mới: sha256, nạp strict =====
from uniav_app.config import Config
from uniav_app.athena_api import checkpoint_info, load_pipeline
PROTO = json.load(open(f'{FUSHION}/configs/final_protocol.json'))
PP = PROTO['uniav']['postprocess']
CKPT = '/content/ckpt/athena_yc2clean.pth'
os.makedirs('/content/ckpt', exist_ok=True)
if not (os.path.exists(CKPT) and sha256(CKPT) == ATHENA_SHA):
    shutil.copy2(ATHENA_CKPT, CKPT)
assert sha256(CKPT) == ATHENA_SHA, 'checkpoint khác sha256 của phần A'
INFO = checkpoint_info(CKPT)
for k in ('run', 'commit', 'epoch', 'epoch_seg', 'has_state_dict_seg', 'n_params_M', 'caption_pool_file'):
    print(f'{k:20s}', INFO[k])
print('final_eval (dev)    ', {k: round(INFO['final_eval'][k], 3) for k in ('R@0.3', 'R@0.5', 'R@0.7', 'mIoU', 'CIDEr')})
assert INFO['run'] == ATHENA_RUN and INFO['has_state_dict_seg'] and (INFO['epoch_seg'], INFO['epoch']) == (8, 10)

common = dict(checkpoint=CKPT, expected_run=ATHENA_RUN, theta=None, save_z=True, feature_source='hf', drive_data=DRIVE_DATA,
              meta_dir=META, video_src=f'{DRIVE_DATA}/videos', omni_subset=SUBSET, cache_root=LOCAL,
              k_max=PP['k_max'], d_min=PP['d_min'], merge_tiou=PP['merge_tiou'])
cfg = Config.colab(tag=f'athena_{ATHENA_H8}', **common)
cfg_tr = Config.colab(tag=f'athena_{ATHENA_H8}_train', **common)
assert list(cfg.theta_grid) == PP['theta']['grid'], 'lưới θ khác protocol'
pipe = load_pipeline(cfg)                                    # state_dict nạp strict=True: sai khoá là lỗi ở đây
print('OK: nạp strict | seg_model', 'riêng (state_dict_seg)' if pipe.seg_model is not pipe.model else 'chung',
      '| device', pipe.device, '| output gallery', cfg.out_dir, '| train', cfg_tr.out_dir)
assert pipe.seg_model is not pipe.model, 'phải cắt đoạn bằng state_dict_seg'

run                  yc2clean_E1
commit               39d8161
epoch                10
epoch_seg            8
has_state_dict_seg   True
n_params_M           66.35
caption_pool_file    caption_emb_iv2j_yc2clean.npz
final_eval (dev)     {'R@0.3': 73.868, 'R@0.5': 54.592, 'R@0.7': 30.789, 'mIoU': 50.711, 'CIDEr': 89.898}
OK: nạp strict | seg_model riêng (state_dict_seg) | device cuda | output gallery /content/uniav_local/athena_961f1a89 | train /content/uniav_local/athena_961f1a89_train


In [4]:
# ===== 4. Video, GT, feature =====
from uniav_app.subset import gt_for_gallery, load_gallery, load_gt_annotations, load_gt_manifest
from uniav_app.features import get_source
gallery = load_gallery(cfg)
splits = {r['video_id']: r['split'] for r in gallery}
durations = {r['video_id']: r['duration'] for r in gallery}
DEV = sorted(v for v, s in splits.items() if s == 'dev')
VAL = sorted(v for v, s in splits.items() if s == 'val')
GT_MANIFEST = load_gt_manifest(cfg.meta_dir)
ANN_ALL = load_gt_annotations(cfg.annotations, subset=None)
gt = gt_for_gallery(gallery, GT_MANIFEST, ANN_ALL)
dur_tr_all = json.load(open(f'{U}/ft_train/durations.json'))   # thời lượng mp4 trainsplit (không phụ thuộc UniAV)
tr_ids = sorted({os.path.splitext(os.path.basename(json.loads(l)['video']))[0]
                 for l in open(f'{META}/train_omni_video.trainsplit.jsonl') if l.strip()} & set(dur_tr_all))
if not DONE:
    src = get_source(cfg)
    rec_g = src.prepare([r['video_id'] for r in gallery], durations)
    rec_t = src.prepare(tr_ids, {v: dur_tr_all[v] for v in tr_ids})
    print('feature: gallery', len(rec_g), '| trainsplit', len(rec_t))
    assert len(rec_g) == 498 and len(rec_t) == 1002, 'thiếu feature'
print('gallery', len(gallery), '(val', len(VAL), '/ dev', len(DEV), ') | GT event', sum(len(g) for g in gt.values()),
      '| trainsplit', len(tr_ids))
assert (len(gallery), len(VAL), len(DEV), len(tr_ids)) == (498, 394, 104, 1002)
assert not set(tr_ids) & set(splits), 'trainsplit chạm gallery'

[gallery] 498 videos from /content/drive/MyDrive/Colab Notebooks/code KL/Fushion/uemr/omni_cache/common/subset_videos.json (read only)
[hf] 498 videos: 0 ready, 498 to prepare -> /content/uniav_local/feats/hf


manifest.json:   0%|          | 0.00/28.8k [00:00<?, ?B/s]

[hf] shard 1/8 iv2_feats_00.tar: 76 videos


iv2_feats/iv2_feats_00.tar: reconstructing file:   0%|          |  0.00B /  270MB            

iv2_feats/iv2_feats_00.tar: downloading bytes:           |  0.00B            

...

iv2_feats/iv2_feats_06.tar: downloading bytes:           |  0.00B            

[hf] shard 8/8 iv2_feats_07.tar: 45 videos


iv2_feats/iv2_feats_07.tar: reconstructing file:   0%|          |  0.00B / 86.8MB            

iv2_feats/iv2_feats_07.tar: downloading bytes:           |  0.00B            

feature: gallery 498 | trainsplit 1002
gallery 498 (val 394 / dev 104 ) | GT event 3803 | trainsplit 1002


In [5]:
# ===== 5. Tái lập final_eval của checkpoint trên dev (trọng số chính, k = #GT, như train.py::evaluate) =====
from uniav_app.sanity import KEYS, _proposals
from uniav_app.metrics import best_tious
if DONE:
    print('bỏ qua (đầu ra đã có)')
else:
    ann_dev = {v: ANN_ALL[v] for v in DEV}
    pipe_main = load_pipeline(cfg, use_seg_weights=False)
    preds, gts = {}, {}
    for v in DEV:
        secs, scores = _proposals(pipe_main, rec_g[v], ann_dev[v]['duration'])
        preds[v] = [(float(a), float(b), float(c)) for (a, b), c in zip(secs, scores)]
        gts[v] = [(e['ts'], e['te']) for e in ann_dev[v]['events']]
    del pipe_main
    best = np.concatenate(list(best_tious(preds, gts, 'K_GT').values()))
    ours = {**{f'R@{t}': 100.0 * float((best >= t).mean()) for t in (0.3, 0.5, 0.7)}, 'mIoU': 100.0 * float(best.mean())}
    PARITY = {k: {'ours': round(ours[k], 2), 'final_eval': round(INFO['final_eval'][k], 2),
                  'diff': round(ours[k] - INFO['final_eval'][k], 2)} for k in KEYS}
    print(json.dumps(PARITY, indent=1))
    assert max(abs(x['diff']) for x in PARITY.values()) <= 1.0, 'uniav_app không tái lập được final_eval (dev): dừng'

{
 "R@0.3": {
  "ours": 73.87,
  "final_eval": 73.87,
  "diff": 0.0
 },
 "R@0.5": {
  "ours": 54.59,
  "final_eval": 54.59,
  "diff": -0.0
 },
 "R@0.7": {
  "ours": 30.79,
  "final_eval": 30.79,
  "diff": -0.0
 },
 "mIoU": {
  "ours": 50.71,
  "final_eval": 50.71,
  "diff": -0.0
 }
}


In [6]:
# ===== 6. Detector (state_dict_seg) cho 498 video gallery: proposal thô + Z =====
from uniav_app.detect import run_detection, timing
if DONE:
    print('bỏ qua (đầu ra đã có)')
else:
    raw_g = run_detection(cfg, pipe, rec_g, splits)
    assert len(raw_g) == 498
    print('thời gian:', timing(raw_g))

[detect] 498 videos: 0 done, 498 to run | seg weights: best_seg | device cuda
[detect] 1/498 -AwyG1JcMp8: n=308 100 proposals | 0.39 s/video
[detect] 2/498 -ErPSunMfcs: n=155 100 proposals | 0.34 s/video
[detect] 3/498 -GlSSp5ZOCQ: n=246 100 proposals | 0.36 s/video
[detect] 100/498 C0gWH2E4lWg: n=358 100 proposals | 0.45 s/video
[detect] 200/498 OSabGgdaQeM: n=240 100 proposals | 0.44 s/video
[detect] 300/498 _mL1gihKDw0: n=555 100 proposals | 0.44 s/video
[detect] 400/498 nfVXBQwOCMc: n=441 100 proposals | 0.44 s/video
[detect] 498/498 zqTXQ-YqrgQ: n=181 100 proposals | 0.44 s/video
thời gian: {'videos': 498, 'device': 'cuda', 'load_features s/video': 0.0028080321285140564, 'detect s/video': 0.4305154618473896, 'save_z s/video': 0.0027279116465863456, 'detect per minute of video (s)': 0.08251399589187641}


In [7]:
# ===== 7. θ theo quy tắc protocol, chỉ trên 104 video dev (val không vào bước này) =====
from uniav_app.evaluate import choose_theta, theta_sweep
from uniav_app.io_utils import write_json
if DONE:
    THETA_INFO = json.load(open(f'{OUT_G}/theta.json'))
else:
    sweep = theta_sweep({v: raw_g[v] for v in DEV}, gt, {v: 'dev' for v in DEV}, cfg)
    display(pd.DataFrame(sweep)[['theta', 'mean_K_pred', 'mean_K_GT', 'AR@K_GT@0.5', 'F1@0.5']].round(3))
    th, on = choose_theta(sweep, 'dev')
    assert on == 'dev'
    THETA_INFO = {'theta': th, 'chosen_on': 'dev', 'rule': PP['theta']['rule'], 'grid': PP['theta']['grid'],
                  'videos': len(DEV), 'checkpoint_sha256': ATHENA_SHA,
                  'theta_old_checkpoint': PP['theta']['value_current_uniav']}
    write_json(cfg.theta_path, THETA_INFO)
    write_json(cfg_tr.theta_path, THETA_INFO)
print('θ:', THETA_INFO)
assert THETA_INFO['chosen_on'] == 'dev'

,theta,mean_K_pred,mean_K_GT,AR@K_GT@0.5,F1@0.5
0,0.05,29.981,7.433,0.561,0.327
1,0.10,29.981,7.433,0.561,0.327
2,0.15,29.269,7.433,0.561,0.333
3,0.20,25.125,7.433,0.561,0.363
4,0.25,18.125,7.433,0.559,0.431
5,0.30,12.788,7.433,0.558,0.491
6,0.35,9.298,7.433,0.543,0.526
7,0.40,6.990,7.433,0.502,0.539
8,0.45,5.106,7.433,0.417,0.497
9,0.50,2.990,7.433,0.279,0.400


θ: {'theta': 0.4, 'chosen_on': 'dev', 'rule': 'dev: mean K_pred closest to mean K_GT, tie -> smaller theta', 'grid': [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6], 'videos': 104, 'checkpoint_sha256': '961f1a895f4b43c811d3e0cfa31448641989a0a0e39e421ed397682407643d58', 'theta_old_checkpoint': 0.4}


In [8]:
# ===== 8. Hậu xử lý (uemr) -> segments_pred.jsonl cho gallery =====
from uniav_app.predict import run_predictions
if DONE:
    print('bỏ qua (đầu ra đã có)')
else:
    pred_g, _ = run_predictions(cfg, pipe, rec_g, raw_g, THETA_INFO['theta'])
    k_dev = np.mean([len(pred_g[v]['events']) for v in DEV]); kgt_dev = np.mean([len(gt[v]) for v in DEV])
    print(f'dev: K_pred trung bình {k_dev:.2f} | K_GT {kgt_dev:.2f}')
    assert abs(k_dev - kgt_dev) < 1.0, 'K_pred dev lệch K_GT quá 1'

[predict] 1/498 | 0.39 s/video
[predict] 100/498 | 0.41 s/video
[predict] 200/498 | 0.41 s/video
[predict] 300/498 | 0.41 s/video
[predict] 400/498 | 0.41 s/video
[predict] 498/498 | 0.41 s/video
[predict] theta=0.4 -> predictions.jsonl, predictions_api.jsonl, segments_pred.jsonl (3689 segments)
dev: K_pred trung bình 6.99 | K_GT 7.43


In [9]:
# ===== 9. Trainsplit (1.002 video): detector + hậu xử lý với cùng θ =====
if DONE:
    print('bỏ qua (đầu ra đã có)')
else:
    raw_t = run_detection(cfg_tr, pipe, rec_t, {v: 'train' for v in tr_ids})
    pred_t, _ = run_predictions(cfg_tr, pipe, rec_t, raw_t, THETA_INFO['theta'])
    assert len(raw_t) == len(pred_t) == 1002

[detect] 1002 videos: 0 done, 1002 to run | seg weights: best_seg | device cuda
[detect] 1/1002 --bv0V6ZjWI: n=333 100 proposals | 0.47 s/video
[detect] 2/1002 -AfxeTnCbVQ: n=441 100 proposals | 0.46 s/video
[detect] 3/1002 -ORd4EBliqw: n=499 100 proposals | 0.47 s/video
[detect] 100/1002 5cqJkRqbHrs: n=382 100 proposals | 0.44 s/video
[detect] 200/1002 CK_GWBMUXGw: n=403 100 proposals | 0.44 s/video
[detect] 300/1002 HbXjlFaPzOM: n=306 100 proposals | 0.44 s/video
[detect] 400/1002 Nbl97Bp3Kzw: n=311 100 proposals | 0.44 s/video
[detect] 500/1002 U10oixMc9_k: n=284 100 proposals | 0.44 s/video
[detect] 600/1002 _bICKhr9AGo: n=345 100 proposals | 0.44 s/video
[detect] 700/1002 fw_3Bg17UFU: n=305 100 proposals | 0.44 s/video
[detect] 800/1002 nfO0YS2dSdw: n=368 100 proposals | 0.44 s/video
[detect] 900/1002 u2U7VkXHtMU: n=273 100 proposals | 0.44 s/video
[detect] 1000/1002 zudeWqL9ALU: n=352 100 proposals | 0.44 s/video
[detect] 1002/1002 zwbCD3CUFIs: n=405 100 proposals | 0.44 s/video


In [10]:
# ===== 10. Kiểm, SOURCE.json, chép lên Drive =====
SUMMARY = {}
for c, ids, out, name in ((cfg, sorted(splits), OUT_G, 'gallery'), (cfg_tr, tr_ids, OUT_T, 'trainsplit')):
    if not DONE:
        lines = [json.loads(l) for l in open(c.segments_path) if l.strip()]
        per = Counter(r['video_id'] for r in lines)
        zs = {os.path.basename(p)[:-4] for p in glob.glob(f'{c.z_dir}/*.npz')}
        assert set(per) == set(ids), f'{name}: video không có đoạn {sorted(set(ids) - set(per))[:5]}'
        assert set(ids) <= zs, f'{name}: thiếu Z {sorted(set(ids) - zs)[:5]}'
        assert all(0 <= r['ts'] < r['te'] and 'conf' in r for r in lines)
        src = {'checkpoint': ATHENA_CKPT, 'sha256': ATHENA_SHA, 'hash8': ATHENA_H8, 'run': INFO['run'],
               'epoch_seg': INFO['epoch_seg'], 'epoch': INFO['epoch'], 'seg_weights': 'state_dict_seg (best_seg, dev R@0.5)',
               'split': name, 'videos': len(ids), 'segments': len(lines), 'theta': THETA_INFO,
               'postprocess': {k: PP[k] for k in ('k_max', 'd_min', 'merge_tiou')}, 'feature_source': 'hf',
               'dev_parity_final_eval': PARITY, 'fushion_commit': COMMIT, 'env': ENV, 'created': time.strftime('%Y-%m-%d %H:%M:%S')}
        write_json(c.out_dir / 'SOURCE.json', src)
        os.makedirs(out, exist_ok=True)
        shutil.copytree(c.out_dir, out, dirs_exist_ok=True)
    s = json.load(open(f'{out}/SOURCE.json'))
    n_z = len(glob.glob(f'{out}/Z/*.npz'))
    SUMMARY[name] = {'dir': out, 'videos': s['videos'], 'segments': s['segments'], 'Z': n_z, 'θ': s['theta']['theta']}
    assert s['sha256'] == ATHENA_SHA and n_z >= len(ids) and s['videos'] == len(ids)
print(json.dumps(SUMMARY, indent=1, ensure_ascii=False))
print('dev parity:', json.load(open(f'{OUT_G}/SOURCE.json'))['dev_parity_final_eval'])
print('-> bước 2: notebooks/uemr_final_AZ_clean.ipynb (runtime A100 mới), UNIAV_SRC =', [OUT_G, OUT_T])

{
 "gallery": {
  "dir": "/content/drive/MyDrive/Colab Notebooks/code KL/Fushion/uemr/uniav_cache/athena_961f1a89",
  "videos": 498,
  "segments": 3689,
  "Z": 498,
  "θ": 0.4
 },
 "trainsplit": {
  "dir": "/content/drive/MyDrive/Colab Notebooks/code KL/Fushion/uemr/uniav_cache/athena_961f1a89_train",
  "videos": 1002,
  "segments": 7802,
  "Z": 1002,
  "θ": 0.4
 }
}
dev parity: {'R@0.3': {'ours': 73.87, 'final_eval': 73.87, 'diff': 0.0}, 'R@0.5': {'ours': 54.59, 'final_eval': 54.59, 'diff': -0.0}, 'R@0.7': {'ours': 30.79, 'final_eval': 30.79, 'diff': -0.0}, 'mIoU': {'ours': 50.71, 'final_eval': 50.71, 'diff': -0.0}}
-> bước 2: notebooks/uemr_final_AZ_clean.ipynb (runtime A100 mới), UNIAV_SRC = ['/content/drive/MyDrive/Colab Notebooks/code KL/Fushion/uemr/uniav_cache/athena_961f1a89', '/content/drive/MyDrive/Colab Notebooks/code KL/Fushion/uemr/uniav_cache/athena_961f1a89_train']
